# 🧩 Cuaderno 03 — Descomposición LU

**Métodos Numéricos · Unidad 3: Sistemas de Ecuaciones Lineales**

Este cuaderno es **taller, no libro**. La teoría completa —las dos etapas, Doolittle contra Crout, la inversa y el condicionamiento— está en el documento:

📖 [`docs/metodos/02_descomposicion_lu.md`](https://github.com/ITH-MGL-MN/U3_Sistemas_de_ecuaciones_lineales/blob/master/docs/metodos/02_descomposicion_lu.md)

Aquí vas a **programar la factorización**.

### 🎯 Lo que vas a entregar

- **Pregunta 9** — `mi_lu(A)`: devuelve **dos** matrices, `return L, U`.
- **Pregunta 10** — `mi_sustitucion_adelante(L, b)`: resuelve $\mathbf{L}\mathbf{y}=\mathbf{b}$.
- **Pregunta 11** — `mi_lu_solve(L, U, b)`: las dos sustituciones y listo.

### ⚙️ Las dos reglas de siempre

1. **Aritmética exacta** con `M.sym`: verás $-3/2$, nunca `-1.5000000000000002`.
2. **Python empieza en 0** y el libro en 1. La tabla de equivalencias está en la sección 2.


In [ ]:
#@title ⚙️ Cargar entorno estilo MATLAB (ejecuta PRIMERO)

import os, sys, subprocess, importlib

# --- Detecta si corremos en Google Colab ---
try:
    import google.colab
    ES_COLAB = True
except ImportError:
    ES_COLAB = False

REPO_URL = "https://github.com/ITH-MGL-MN/U3_Sistemas_de_ecuaciones_lineales.git"


def _buscar_raiz(base, saltos=6):
    """Sube desde `base` hasta la carpeta que tiene config/config.yaml."""
    b = base
    for _ in range(saltos):
        if os.path.exists(os.path.join(b, "config", "config.yaml")):
            return os.path.abspath(b)
        padre = os.path.dirname(b)
        if padre == b:
            break
        b = padre
    return None


if ES_COLAB:
    # ============================================================
    # Google Colab: montar Drive y traer el repositorio
    # ============================================================
    # El repo se clona DENTRO de tu Drive, no en /content (que se borra al
    # cerrar la sesión). Así sobrevive entre sesiones y `git pull` solo
    # actualiza lo que cambió.
    from google.colab import drive

    drive.mount("/content/drive")

    BASE = "/content/drive/MyDrive/CursoMN"
    REPO = os.path.join(BASE, "U3_Repo")
    os.makedirs(BASE, exist_ok=True)

    try:
        if not os.path.exists(os.path.join(REPO, ".git")):
            print("Clonando el repositorio (la primera vez tarda un poco)...")
            subprocess.run(["git", "clone", REPO_URL, REPO], check=True)
        else:
            print("Actualizando el repositorio...")
            subprocess.run(["git", "-C", REPO, "pull", "--ff-only"], check=True)
        print("✅ Repositorio listo en", REPO)
    except Exception as _exc:
        print("⚠️ No pude clonar/actualizar el repositorio:", _exc)
        print("   Si la carpeta ya existe en tu Drive, puedes continuar.")
else:
    # En tu computadora el repo ya está en disco: solo hay que encontrarlo.
    _ruta_nb = globals().get("__vsc_ipynb_file__")      # VS Code
    REPO = _buscar_raiz(os.path.dirname(os.path.abspath(_ruta_nb))) if _ruta_nb else None
    REPO = REPO or _buscar_raiz(os.getcwd())

if REPO is None:
    raise RuntimeError(
        "No encuentro la carpeta del repositorio (la que tiene config/config.yaml).\n"
        "Abre el cuaderno desde dentro de U3_Repo o revisa que el clon de Colab exista."
    )

print("Repositorio :", REPO)

# ============================================================
# Biblioteca del curso (lib/robomat.py)
# ============================================================
# En sys.path van las DOS raíces que hacen falta:
#   REPO/lib    -> robomat.py y el paquete mn_motor/
#   REPO        -> el paquete preguntas/ (las preguntas de este cuaderno)
for _ruta in (os.path.join(REPO, "lib"), REPO):
    if _ruta not in sys.path:
        sys.path.insert(0, _ruta)

import robomat as _robomat
importlib.reload(_robomat)          # recoge los cambios si editaste la libreria
from robomat import *               # noqa: F401,F403

print("✅ Entorno cargado desde:", _robomat.__file__)


# ============================================================
# Motor de calificación
# ============================================================
# En Colab SIEMPRE se usa el blob ofuscado. En tu computadora puedes escoger
# "modulos": importa lib/mn_motor/ tal cual, y así funcionan F12, el
# autocompletado y los puntos de ruptura. Es el modo para depurar.
MODO_MOTOR = "modulos"

MODO = "blob" if ES_COLAB else MODO_MOTOR

if MODO == "modulos":
    try:
        # Se borran del caché para que una edición en lib/mn_motor/ o en
        # preguntas/ se note al volver a correr la celda, sin reiniciar.
        for _n in [n for n in sys.modules
                   if n.split(".")[0] in ("mn_motor", "preguntas")]:
            del sys.modules[_n]
        # Import ESTÁTICO después del borrado: es lo que Pylance analiza.
        import mn_motor as _motor
        from mn_motor import *        # noqa: F401,F403
    except ImportError as _exc:
        # En un clon del alumno no existe preguntas/: no hay clave de
        # respuestas. Sin problema: el blob trae lo mismo.
        print("ℹ️ No pude importar el motor del repositorio:", _exc)
        print("   Uso el motor ofuscado del blob.")
        MODO = "blob"

if MODO == "blob":
    import base64 as _b64
    import zlib as _zlib

    # Cada cuaderno tiene su propio blob: aquí va el del Cuaderno 03.
    ruta_motor = os.path.join(REPO, "grader", "u3_03_lu.txt")
    with open(ruta_motor, encoding="utf-8") as _f:
        _blob = _f.read().strip()
    _codigo = _zlib.decompress(_b64.b64decode(_blob)).decode("utf-8")
    # _REPO_U3 le dice al motor dónde está el repo: dentro del blob no existe
    # __file__, así que no puede deducirlo solo.
    os.environ["U3_REPO"] = REPO
    _MOTOR = {"__name__": "blob_u3"}
    exec(_codigo, _MOTOR)
    globals().update({_n: _MOTOR[_n] for _n in _MOTOR["__all__"]})


# Este cuaderno califica SU conjunto de preguntas (carpeta preguntas/u3_03_lu).
# En modo blob ya viene incrustado, pero en modo módulos hay que declararlo.
usar_cuaderno("u3_03_lu", "U3_03_Descomposicion_LU")

print(f"✅ Motor de calificación cargado ({MODO}) · cuaderno {cuaderno()}")
print("   Sigue con la celda de identificación.")


## 0 · Tu identificación

Todo lo que sigue usa **tus** datos. Al ejecutar la celda de abajo, Colab te
pide permiso para leer tu correo; de ahí salen tu número de control y tu nombre.
Con eso se siembran los números de tus preguntas: **cada alumno recibe datos
distintos**.

Fuera de Colab no hay sesión de Google, así que el correo se escribe a mano en
la celda.


In [ ]:
#@title 👤 Identificación automática (correo institucional)
# ============================================================
# Uso el correo con el que montaste Drive: de ahí salen tu NC y tu nombre.
# ============================================================
if ES_COLAB:
    import requests
    import google.auth
    import google.auth.transport.requests
    from google.colab import auth

    try:
        auth.authenticate_user()
        creds, _ = google.auth.default()
        creds.refresh(google.auth.transport.requests.Request())
        info = requests.get(
            "https://www.googleapis.com/drive/v3/about?fields=user",
            headers={"Authorization": "Bearer " + creds.token},
            timeout=20,
        ).json()
        cuenta = info.get("user", {})
        email = cuenta.get("emailAddress", "")
        nombre = cuenta.get("displayName", "")
    except Exception as e:
        print("⚠️ No pude obtener tu correo automáticamente.")
        print("   Vuelve a ejecutar y acepta el permiso en la ventana de Google.")
        print("   Detalle:", e)
        email, nombre = "", ""
else:
    # Modo local: aquí no hay sesión de Google, escribe tu correo.
    email = "m16330887@hermosillo.tecnm.mx"      # <-- cambia por el tuyo
    nombre = ""                                 # Google solo lo da en Colab

if email and email.lower().endswith(".tecnm.mx"):
    alumno_id = email
    print("✅ Identificado:", email)
    print("   Número de control:", extraer_nc(email))
    print("   Nombre:", nombre if nombre else "(no disponible fuera de Colab)")
else:
    from IPython.display import HTML, display as _display

    alumno_id = None
    _display(HTML(
        '<p style="color:#b00020;background:#ffe0e0;padding:12px;border-radius:8px;'
        'font-weight:bold">⛔ Necesito tu correo institucional (@….tecnm.mx).<br>'
        'Si iniciaste sesión con otra cuenta, ciérrala o usa una ventana de '
        'incógnito.</p>'))
    print("Correo detectado:", repr(email))

if alumno_id is None:
    raise RuntimeError("Detente: sin correo institucional no puedo generar tu tarea.")

# La tarea se genera UNA sola vez: todas las celdas de preguntas llaman a
# pregunta(n) sobre este MI_TAREA, así que todas hablan de lo mismo.
MI_TAREA = generar_tarea(alumno_id, nombre)
print(f"\nListo: {len(MI_TAREA['preguntas'])} preguntas para el NC {MI_TAREA['nc']}.")


## 1 · La idea: factorizar una vez, resolver muchas

La eliminación gaussiana del Cuaderno 02 hace un trabajo que **no depende de
$\mathbf{b}$**: los multiplicadores y los pivotes salen solo de $\mathbf{A}$. Si
hay que resolver el mismo sistema con diez lados derechos distintos, esa cuenta
se repite diez veces sin necesidad.

LU la guarda. Se escribe $\mathbf{A} = \mathbf{L}\mathbf{U}$ y el sistema se
parte en dos:

$$\mathbf{A}\mathbf{x} = \mathbf{b} \;\Longleftrightarrow\; \mathbf{L}\underbrace{\mathbf{U}\mathbf{x}}_{\mathbf{y}} = \mathbf{b} \;\Longleftrightarrow\; \mathbf{L}\mathbf{y} = \mathbf{b} \;\text{y luego}\; \mathbf{U}\mathbf{x} = \mathbf{y}$$

Donde $\mathbf{L}$ es triangular inferior con unos en la diagonal y $\mathbf{U}$
es triangular superior. Lo importante es que **$\mathbf{L}$ es exactamente el
registro de la eliminación**: cada $l_{ik}$ es el multiplicador $m_{ik}$ del paso
que anuló la posición $(i,k)$, y $\mathbf{U}$ es la matriz ya triangularizada.

> 📖 Las dos etapas están en la [sección 2.2](https://github.com/ITH-MGL-MN/U3_Sistemas_de_ecuaciones_lineales/blob/master/docs/metodos/02_descomposicion_lu.md#sec-2-2) y la factorización paso a paso en la [sección 2.3](https://github.com/ITH-MGL-MN/U3_Sistemas_de_ecuaciones_lineales/blob/master/docs/metodos/02_descomposicion_lu.md#sec-2-3) del documento.

Seguimos con la misma matriz de todo el curso, la de
$\mathbf{x} = [2,\ 3,\ -1]$:


In [ ]:
#@title La matriz del cuaderno, con aritmética exacta
A = M.sym[[2, 1, -1], [-3, -1, 2], [-2, 1, 2]]
b = M.sym[[8], [-11], [-3]]

mostrar(A)
mostrar(b)
print("tamaño:", A.shape, "· ¿exacto?", es_exacto(A))
print("solución esperada:  x1 = 2,  x2 = 3,  x3 = -1")


## 2 · Pseudocódigo: factorización de Doolittle

La cuenta es **la misma eliminación** del Cuaderno 02, con dos cambios:

- se elimina sobre $\mathbf{A}$ (sin la columna de $\mathbf{b}$), y
- cada multiplicador se **guarda** en $\mathbf{L}$ en vez de tirarse.

```
L = identidad(n)                    # unos en la diagonal, ceros fuera
U = copia de A

para k = 1, ..., n-1:               # columna del pivote
    para i = k+1, ..., n:           # renglones debajo
        m = U[i][k] / U[k][k]
        L[i][k] = m                 # <-- aquí está toda la diferencia
        para j = k, ..., n:
            U[i][j] = U[i][j] - m * U[k][j]

devolver L, U
```

Al final, $\mathbf{U}$ es la matriz triangularizada y $\mathbf{L}$ tiene los
multiplicadores. Compruébalo siempre así: **$\mathbf{L}\mathbf{U}$ tiene que dar
$\mathbf{A}$**.

### Del libro a Python

Los índices coinciden, solo cambian los extremos de los ciclos:

| | Libro | Python |
|---|---|---|
| renglones y columnas | $1\ldots n$ | `0 … n-1` |
| ciclo externo | `k = 1 … n-1` | `for k in range(n - 1)` |
| renglones debajo | `i = k+1 … n` | `for i in range(k + 1, n)` |
| columnas a restar | `j = k … n` | `for j in range(k, n)` |

> ⚠️ Fíjate que aquí el ciclo de `j` llega hasta `n`, no hasta `n+1`: en LU no
> hay columna de $\mathbf{b}$, y esa es justo la razón de que factorizar sirva
> para **cualquier** lado derecho.


## 3 · Tu turno: la factorización

Escribe `mi_lu(A)` en la celda de abajo. Tiene que devolver **dos** matrices: la
llamada es `return L, U`.

`practica('prog_lu')` la prueba con la matriz de la sección 1 y te dice si ya
coincide. No gasta intentos.

Tres trampas:

- $\mathbf{L}$ empieza como **identidad**, no como ceros. La biblioteca ya la tiene: `eye_sym(n)`.
- Para copiar $\mathbf{A}$ sin destruirla, `M.sym[A.tolist()]` (ojo: `M.sym[A]` con un solo corchete no hace lo que parece).
- Tiene que servir para cualquier $n$: la pregunta 9 la prueba con matrices de **2, 3 y 4**. Si escribes los índices fijos a un 3×3, pierdes los otros dos casos.


In [ ]:
#@title ✍️ Escribe tu factorización LU
# Dos ayudas de la biblioteca:
#   eye_sym(n)          -> identidad simbólica (para arrancar la L)
#   M.sym[A.tolist()]   -> copia exacta y modificable de una matriz
# Ojo: M.sym[[...] for i in range(n)] es un SyntaxError de Python. Usa eye_sym.
def mi_lu(A):
    """Devuelve (L, U) con A = L @ U, L unitaria inferior y U superior."""
    raise NotImplementedError("Traduce aquí el pseudocódigo de la sección 2")


practica('prog_lu')


In [ ]:
#@title 📝 Pregunta 9 (vale 1 punto)
pregunta(9)


## 4 · Sustitución hacia adelante

Con $\mathbf{L}\mathbf{y} = \mathbf{b}$ y $\mathbf{L}$ triangular **inferior**,
la primera componente se despeja sola y las demás van cayendo de arriba hacia
abajo:

$$y_i = \frac{1}{l_{ii}}\left(b_i - \sum_{j=1}^{i-1} l_{ij}\,y_j\right), \qquad i = 1, 2, \ldots, n$$

```
para i = 1, ..., n:
    suma = 0
    para j = 1, ..., i-1:           # columnas a la IZQUIERDA de la diagonal
        suma = suma + L[i][j] * y[j]
    y[i] = (b[i] - suma) / L[i][i]
```

Es la sustitución hacia atrás **leída al revés**: mismo ciclo, misma suma, pero
recorriendo de arriba hacia abajo y sumando las columnas de la izquierda.

> 💡 En Doolittle todos los $l_{ii}=1$, así que esa división no hace nada. **No
> lo supongas en el código:** la pregunta 10 prueba una $\mathbf{L}$ que no
> tiene unos en la diagonal.


In [ ]:
#@title ✍️ Escribe tu sustitución hacia adelante
def mi_sustitucion_adelante(L, b):
    """Resuelve L y = b y devuelve y como columna."""
    raise NotImplementedError("Traduce aquí el pseudocódigo de la sección 4")


practica('prog_adelante')


In [ ]:
#@title 📝 Pregunta 10 (vale 1 punto)
pregunta(10)


## 5 · El resolvedor completo

Ya tienes las dos piezas. Resolver con LU es encadenarlas:

```
LU_solve(L, U, b):
    y = sustitucion_hacia_adelante(L, b)    # L y = b
    x = sustitucion_hacia_atras(U, y)       # U x = y
    devolver x
```

La sustitución hacia atrás es **la misma función del Cuaderno 02**
(`sustitucion_atras`), porque $\mathbf{U}$ es triangular superior igual que
antes. Escríbela otra vez aquí (o cópiala) y úsala.


In [ ]:
#@title ✍️ Escribe el resolvedor LU
def _sustitucion_atras(U, y):
    """Despeja x de U x = y (U triangular superior)."""
    n = U.rows
    x = [0] * n
    for i in range(n - 1, -1, -1):
        suma = 0
        for j in range(i + 1, n):
            suma = suma + U[i, j] * x[j]
        x[i] = (y[i, 0] - suma) / U[i, i]
    return M.sym[[x]].T


def mi_lu_solve(L, U, b):
    """Resuelve A x = b usando la factorización ya hecha."""
    raise NotImplementedError("Encadena las dos sustituciones")


practica('prog_lu_solve')


In [ ]:
#@title 📝 Pregunta 11 (vale 1 punto)
pregunta(11)


## 6 · ¿De verdad ahorra? Los números

Factorizar cuesta lo mismo que una eliminación, y resolver después cada sistema
cuesta $O(n^2)$. Con un solo $\mathbf{b}$ eso es **más trabajo**, no menos:

| | $n = 3$ | $n = 10$ | $n = 100$ |
|---|---|---|---|
| dos eliminaciones completas | 18 | 667 | 666 667 |
| factorizar + 2 sustituciones | 27 | 533 | 353 333 |

(multiplicaciones, contando $n^3/3$ para eliminar y $n^2$ por sustitución)

La factorización empieza a ganar cuando $n > 6$. Y si en vez de dos lados
derechos hay muchos, gana siempre: el costo de factorizar **se paga una vez**.

La celda de abajo lo comprueba con dos vectores $\mathbf{b}$ distintos sobre la
misma $\mathbf{A}$: se factoriza una sola vez y se resuelven los dos.


In [ ]:
#@title 💰 Factorizar una vez, resolver dos veces
b2 = M.sym[[1], [2], [3]]

try:
    L, U = mi_lu(A)
    mostrar(L)
    mostrar(U)
    print("¿L @ U = A?", (L @ U) == A)

    print("\ncon b  =", b.T.tolist())
    mostrar(mi_lu_solve(L, U, b))

    print("\ncon b2 =", b2.T.tolist())
    mostrar(mi_lu_solve(L, U, b2))
    print("A no se volvió a tocar: solo se usaron L y U.")
except NotImplementedError:
    print("⏳ Escribe primero mi_lu (sección 3) y mi_lu_solve (sección 5).")


## 7 · Preguntas de teoría (1 a 8)

Las cinco primeras son de opción múltiple. La 6 pide una matriz (la $\mathbf{L}$
de un sistema), la 7 un pivote y la 8 una componente del vector intermedio.


In [ ]:
#@title 🧠 Preguntas 1 a 5 · opción múltiple
pregunta(1)
pregunta(2)
pregunta(3)
pregunta(4)
pregunta(5)


In [ ]:
#@title ✍️ Preguntas 6 a 8 · con tus matrices
# La 6 es una matriz (escríbela como [[1,2,3],[4,5,6],[7,8,9]]);
# la 7 y la 8 son un solo número.
pregunta(6)
pregunta(7)
pregunta(8)


## 8 · Calificar y enviar

- `calificar()` **no envía nada**: úsala las veces que quieras mientras trabajas. En las de programar te dice **qué caso** falló.
- `enviar(alumno_id)` sí manda la calificación y **gasta uno de tus 5 intentos**.


In [ ]:
#@title 📝 Calificar (sin enviar) — úsala las veces que quieras
calificar()


In [ ]:
#@title 📤 Enviar la calificación (máximo 5 intentos)
enviar(alumno_id)


In [ ]:
#@title 🔎 Comprobar lo que ya está guardado (no envía nada)
consultar_calificacion()


---
### ✅ ¿Qué sigue?

En el **Cuaderno 04** vienen las matrices con estructura —tridiagonales con el
algoritmo de Thomas y simétricas con Cholesky— y los **métodos iterativos**
(Jacobi, Gauss-Seidel), que en vez de factorizar se acercan a la solución paso a
paso.

📖 Teoría de este cuaderno:
[`docs/metodos/02_descomposicion_lu.md`](https://github.com/ITH-MGL-MN/U3_Sistemas_de_ecuaciones_lineales/blob/master/docs/metodos/02_descomposicion_lu.md)

📤 Antes de cerrar: ejecuta `calificar()` y luego `enviar(alumno_id)`. Recuerda
que tienes **5 intentos**.
